# PathSnap — Model: K-Nearest Neighbours (KNN)

**Family:** Instance-based model

Stores the training rows and, for a new candidate, looks at the k most similar training rows (in scaled feature space) and lets them vote. No real 'training' step, but prediction is slow on large data and it needs scaled features.

**Task:** for each (GPS observation, candidate road arc) row, predict the probability that the
candidate is the correct road (`label = 1`). For each GPS point, the candidate with the highest
probability is the model's choice (**Top-1**).

**Data**
- Training: `training_data_1.csv` — 80 training trips (all model fitting and cross-validation happen here)
- Test: `student_test_data.csv` — 10 student-test trips, **used exactly once, at the end**

**How leakage is prevented**
1. Train and test trips are checked to be completely separate (no shared `trip_id`).
2. Cross-validation uses **GroupKFold by `trip_id`**, so a trip is never split across train and validation folds.
3. Imputation and scaling live **inside the sklearn Pipeline**, so they are fitted on training folds only.
4. IDs (`trip_id`, `gps_index`, `candidate_arc_id`) and the label are never used as features.
5. Hyperparameters are fixed in advance (same as the team's `models/train_model.py`); nothing is chosen using the test set.
6. The test file is loaded only in the final section, after the model is fully trained.

## 1. Setup

In [ ]:
# Colab already has numpy, pandas, scikit-learn, matplotlib and xgboost.
import os, json, time, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             log_loss, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay)
from sklearn.neighbors import KNeighborsClassifier

warnings.filterwarnings("ignore", category=UserWarning)
SEED = 42
np.random.seed(SEED)
MODEL_NAME = "knn"
print("Model:", MODEL_NAME)

## 2. Find the data files

Put `training_data_1.csv` and `student_test_data.csv` in the **PathSnap** folder in My Drive.
The cell below looks there first, then searches the rest of My Drive (names with a prefix or
a copy suffix like `1de060fd-student_test_data.csv` or `training_data_1 (1).csv` are accepted).
Results (metrics, predictions, model file, plots) are saved to `PathSnap/results/`.

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/PathSnap"   # first place to look
TRAIN_FILE = "training_data_1.csv"
TEST_FILE = "student_test_data.csv"

try:
    from google.colab import drive
    drive.mount("/content/drive")
    SEARCH_ROOT = "/content/drive/MyDrive"
except ImportError:                     # running outside Colab
    DRIVE_DIR = os.environ.get("PATHSNAP_NB_DATA", ".")
    SEARCH_ROOT = DRIVE_DIR

def locate(exact_name, keyword, exclude, max_depth=4):
    """Exact name in DRIVE_DIR first; otherwise search My Drive for a CSV containing `keyword`."""
    direct = os.path.join(DRIVE_DIR, exact_name)
    if os.path.exists(direct):
        return direct
    matches = []
    base_depth = SEARCH_ROOT.rstrip("/").count("/")
    for root, dirs, files in os.walk(SEARCH_ROOT):
        if root.count("/") - base_depth >= max_depth:
            dirs[:] = []
        for f in files:
            low = f.lower()
            if low.endswith(".csv") and keyword in low and not any(x in low for x in exclude):
                matches.append(os.path.join(root, f))
    if not matches:
        print(f"Could not find {exact_name}. Files in {DRIVE_DIR}:",
              sorted(os.listdir(DRIVE_DIR)) if os.path.isdir(DRIVE_DIR) else "(folder does not exist)")
        raise FileNotFoundError(f"Upload {exact_name} to {DRIVE_DIR}")
    # prefer files inside DRIVE_DIR, then the exact name, then the shortest name
    matches.sort(key=lambda m: (not m.startswith(DRIVE_DIR), os.path.basename(m).lower() != exact_name, len(m)))
    if len(matches) > 1:
        print(f"Several candidates for {exact_name}; using the first:", *matches[:5], sep="\n  ")
    return matches[0]

TRAIN_PATH = locate(TRAIN_FILE, "training_data_1", exclude=("handoff",))
TEST_PATH = locate(TEST_FILE, "student_test_data", exclude=("diagnostic",))
OUT_DIR = os.path.join(os.path.dirname(TRAIN_PATH), "results")
os.makedirs(OUT_DIR, exist_ok=True)
print("Training file:", TRAIN_PATH)
print("Test file:    ", TEST_PATH)
print("Results to:   ", OUT_DIR)

## 3. Load the training data (test data is NOT loaded yet)

`trip_id` is read as text and padded to 8 digits so IDs like `00000001` keep their leading zeros.

In [ ]:
META_COLS = ["trip_id", "gps_index", "candidate_arc_id"]
TARGET = "label"
FEATURES = ["distance", "bearing_difference", "speed", "heading_change", "segment_length"]
# Accept the long column names from features/feature_extraction.py too
RENAME = {"distance_m": "distance", "bearing_difference_deg": "bearing_difference",
          "speed_mps": "speed", "heading_change_deg": "heading_change",
          "segment_length_m": "segment_length"}

def load_table(path):
    df = pd.read_csv(path, dtype={"trip_id": str}).rename(columns=RENAME)
    missing = [c for c in META_COLS + FEATURES + [TARGET] if c not in df.columns]
    assert not missing, f"{path}: missing columns {missing}"
    df["trip_id"] = df["trip_id"].str.zfill(8)
    df[FEATURES] = df[FEATURES].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    df[TARGET] = df[TARGET].astype(int)
    return df

train = load_table(TRAIN_PATH)
print(f"Training rows: {len(train):,} | trips: {train.trip_id.nunique()} | "
      f"GPS points: {train.groupby(['trip_id', 'gps_index']).ngroups:,}")

## 4. Training-data checks

In [ ]:
assert set(train[TARGET].unique()) <= {0, 1}, "labels must be 0/1"
assert not train.duplicated(META_COLS).any(), "duplicate (trip, gps_index, arc) rows"
pos_per_point = train.groupby(["trip_id", "gps_index"])[TARGET].sum()
assert (pos_per_point <= 1).all(), "a GPS point has more than one correct arc"
assert not any(c in FEATURES for c in META_COLS + [TARGET]), "ID or label used as a feature"

print("Positive (correct-road) rows: {:.1%}".format(train[TARGET].mean()))
print("GPS points with a correct candidate: {:.1%}".format((pos_per_point == 1).mean()))
print("\nMissing values per feature (undefined values: first GPS points, stationary vehicle):")
print(train[FEATURES].isna().mean().map("{:.2%}".format).to_string())
print("\nCandidates per GPS point:", train.groupby(["trip_id", "gps_index"]).size().describe()[["mean", "50%", "max"]].round(2).to_dict())

## 5. Model pipeline

**Hyperparameters (fixed, same as `models/train_model.py`):** `n_neighbors=15, weights='distance'`

Steps: median imputation with a missing-value indicator → standard scaling → K-Nearest Neighbours (KNN).
Everything is inside one Pipeline, so each preprocessing step is fitted on training data only.

In [ ]:
def make_model():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler()),
        ("clf", KNeighborsClassifier(n_neighbors=15, weights='distance', n_jobs=-1)),
    ])

def prepare_for_fit(model, y_train):
    # XGBoost has no class_weight: balance classes using the TRAINING labels only
    if "clf__scale_pos_weight" in model.get_params():
        pos = int(y_train.sum())
        model.set_params(clf__scale_pos_weight=(len(y_train) - pos) / max(pos, 1))
    return model

make_model()

## 6. Metrics

- **Accuracy / Precision / Recall / F1 (binary)** — per candidate row, threshold 0.5, positive class = correct road
- **Macro F1** — average of the F1 for class 0 and class 1 (required in the report)
- **Log loss** — how good the predicted probabilities are (lower is better; matters for the HMM)
- **ROC AUC** — ranking quality across all thresholds
- **Top-1 accuracy** — for each GPS point, is the highest-probability candidate the correct road? (the main map-matching metric)

In [ ]:
def top1_accuracy(meta, y_true, proba):
    d = meta[["trip_id", "gps_index"]].copy().reset_index(drop=True)
    d["y"] = np.asarray(y_true)
    d["p"] = np.asarray(proba)
    best = d.loc[d.groupby(["trip_id", "gps_index"])["p"].idxmax()]
    has_true = d.groupby(["trip_id", "gps_index"])["y"].max()
    return float(best["y"].sum() / max(int(has_true.sum()), 1))

def compute_metrics(meta, y_true, proba, threshold=0.5):
    y_pred = (proba >= threshold).astype(int)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "log_loss": log_loss(y_true, np.clip(proba, 1e-7, 1 - 1e-7), labels=[0, 1]),
        "roc_auc": roc_auc_score(y_true, proba) if len(np.unique(y_true)) > 1 else np.nan,
        "top1_accuracy": top1_accuracy(meta, y_true, proba),
    }

## 7. Cross-validation on the training trips only (GroupKFold by trip)

In [ ]:
N_SPLITS = 5
X, y, groups = train[FEATURES], train[TARGET], train["trip_id"]
cv_rows = []
for fold, (tr_idx, va_idx) in enumerate(GroupKFold(n_splits=N_SPLITS).split(X, y, groups)):
    tr_trips, va_trips = set(groups.iloc[tr_idx]), set(groups.iloc[va_idx])
    assert tr_trips.isdisjoint(va_trips), "trip leakage between CV folds"
    model = prepare_for_fit(make_model(), y.iloc[tr_idx])
    t0 = time.perf_counter()
    model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    fit_s = time.perf_counter() - t0
    t0 = time.perf_counter()
    proba = model.predict_proba(X.iloc[va_idx])[:, 1]
    pred_s = time.perf_counter() - t0
    m = compute_metrics(train.iloc[va_idx], y.iloc[va_idx].to_numpy(), proba)
    m.update(fold=fold, train_trips=len(tr_trips), val_trips=len(va_trips), fit_s=fit_s, predict_s=pred_s)
    cv_rows.append(m)
    print(f"fold {fold}: top1={m['top1_accuracy']:.4f} macroF1={m['macro_f1']:.4f} "
          f"logloss={m['log_loss']:.4f} ({fit_s:.1f}s fit, {va_trips and len(va_trips)} val trips)")

cv = pd.DataFrame(cv_rows)
cv.to_csv(os.path.join(OUT_DIR, f"{MODEL_NAME}_cv_folds.csv"), index=False)
metric_cols = ["accuracy", "precision", "recall", "f1", "macro_f1", "log_loss", "roc_auc", "top1_accuracy"]
cv_summary = cv[metric_cols].agg(["mean", "std"]).T
cv_summary

## 8. Train the final model on all 80 training trips

In [ ]:
final_model = prepare_for_fit(make_model(), y)
t0 = time.perf_counter()
final_model.fit(X, y)
final_fit_s = time.perf_counter() - t0
joblib.dump(final_model, os.path.join(OUT_DIR, f"{MODEL_NAME}.joblib"))
print(f"Trained on {train.trip_id.nunique()} trips, {len(train):,} rows in {final_fit_s:.1f}s")

## 9. Final evaluation on the student-test trips (used once)

The test file is loaded only now. The trained model is applied as-is: nothing is refitted or tuned.

In [ ]:
test = load_table(TEST_PATH)
shared = set(train.trip_id) & set(test.trip_id)
assert not shared, f"LEAKAGE: trips in both train and test: {sorted(shared)}"
print(f"Test rows: {len(test):,} | trips: {test.trip_id.nunique()} | "
      f"GPS points: {test.groupby(['trip_id', 'gps_index']).ngroups:,} | shared trips with train: 0")

t0 = time.perf_counter()
test_proba = final_model.predict_proba(test[FEATURES])[:, 1]
test_pred_s = time.perf_counter() - t0
test_metrics = compute_metrics(test, test[TARGET].to_numpy(), test_proba)
test_metrics["inference_time_s"] = test_pred_s
test_metrics["inference_ms_per_1k_rows"] = 1000 * test_pred_s / len(test) * 1000
pd.Series(test_metrics).round(4)

### Per-trip Top-1 accuracy on the student-test trips

In [ ]:
per_trip = []
for tid, g in test.assign(p=test_proba).groupby("trip_id"):
    per_trip.append({"trip_id": tid, "gps_points": g.groupby("gps_index").ngroups,
                     "top1_accuracy": top1_accuracy(g, g[TARGET].to_numpy(), g["p"].to_numpy())})
per_trip = pd.DataFrame(per_trip).sort_values("top1_accuracy")
per_trip.to_csv(os.path.join(OUT_DIR, f"{MODEL_NAME}_test_per_trip.csv"), index=False)
per_trip

## 10. Plots

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
ConfusionMatrixDisplay(confusion_matrix(test[TARGET], (test_proba >= 0.5).astype(int)),
                       display_labels=["wrong road (0)", "correct road (1)"]).plot(ax=ax, colorbar=False)
ax.set_title(f"{MODEL_NAME} — student-test confusion matrix")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f"{MODEL_NAME}_plots.png"), dpi=150)
plt.show()
print("KNN has no built-in feature importance.")

## 11. Save results

- `<model>_summary.json` / `model_results_<model>.csv` — CV mean/std and the student-test metrics
- `<model>_test_predictions.csv` — probability for every test candidate (the HMM's emission input)
- `<model>.joblib` — the trained model

In [ ]:
row = {"model": MODEL_NAME, "train_trips": train.trip_id.nunique(), "test_trips": test.trip_id.nunique(),
       "cv": f"GroupKFold(n_splits={N_SPLITS}, groups=trip_id)", "final_fit_s": final_fit_s}
for k in metric_cols:
    row[f"cv_{k}_mean"] = cv[k].mean()
    row[f"cv_{k}_std"] = cv[k].std()
for k, v in test_metrics.items():
    row[f"test_{k}"] = v
pd.DataFrame([row]).to_csv(os.path.join(OUT_DIR, f"model_results_{MODEL_NAME}.csv"), index=False)
with open(os.path.join(OUT_DIR, f"{MODEL_NAME}_summary.json"), "w") as fh:
    json.dump(row, fh, indent=2, default=float)

test[META_COLS + [TARGET]].assign(probability=test_proba).to_csv(
    os.path.join(OUT_DIR, f"{MODEL_NAME}_test_predictions.csv"), index=False)
print("Saved to", OUT_DIR)

## 12. Comparison with the other models run so far

Reads every `model_results_*.csv` in the results folder (each of the five notebooks writes one).

In [ ]:
frames = [pd.read_csv(p) for p in sorted(glob.glob(os.path.join(OUT_DIR, "model_results_*.csv")))]
assert frames, "No model_results_*.csv yet: run sections 1-11 first"
cols = ["model", "cv_top1_accuracy_mean", "cv_macro_f1_mean", "cv_log_loss_mean",
        "test_top1_accuracy", "test_macro_f1", "test_log_loss", "test_roc_auc", "final_fit_s", "test_inference_time_s"]
comparison = pd.concat(frames, ignore_index=True)[cols].sort_values("test_top1_accuracy", ascending=False)
comparison.round(4)

## 13. Observations (write after running — from the numbers above only)

- CV vs student-test: are the scores similar (good generalisation) or is test much lower?
- Which trips have the lowest Top-1 accuracy, and what do they have in common?
- Which features matter most?
- Strengths / weaknesses of this model for map-matching: